# Notebook 3: Certified top-k sampling (decision-exact)

**Run on:** Colab T4 or better. No model needed; section 7 replays real logits saved by Notebook 2.

Notebook 1 showed that top-k sampling is fast but changes the sampler's decisions unless the missing tail is accounted for. This notebook goes one step further: a sampler that **provably makes the same commit and stop decisions as the full sampler whenever its entropy bounds are tight enough, falling back to full-vocabulary computation for the slots where they are not**, while touching only the top-k tokens for almost every slot. The main implementation samples tokens from the top-k only; section 4 tests an optional tail branch that would make sampling exact too.

**The idea in three parts**

1. **Entropy intervals, not estimates.** For each slot, top-k plus one cheap log-sum-exp pass gives a guaranteed range [lo, hi] that contains the true entropy.
    - hi = top-k entropy + (-m * log m) + m * log(V - k), the tail spread evenly
    - lo = top-k entropy + m * (-log(min(m, p_k))), where p_k is the smallest kept probability. Every tail token is at most as likely as p_k, which makes this much tighter than the simple -m log m bound.
2. **Certify each decision.** The commit rule and the stopping rule only need to know which side of a threshold the entropies fall on. If every entropy inside the intervals leads to the same decision, the decision is certified exact. Only the few slots whose intervals straddle a threshold get their exact entropy computed (one full-vocabulary row each). This step is called *refinement*.
3. **Optional: exact sampling via a tail branch.** Draw u from Uniform(0, 1). If u < m (the tail mass), sample from the tail using that slot's full row; otherwise sample from the top-k. Each token then has exactly its true probability, and the full row is needed only with probability m, which is tiny for confident slots.

The self-conditioning mix keeps only the top-k terms; its error is bounded by m * max(||embedding||), which is reported below.

**What makes this paper-worthy:** a guarantee instead of "accuracy looked fine." The key number to measure on real logits is the **refinement rate**: the fraction of slots that need a full row. If it is around 1%, the sampler costs about 1% of the full version while being decision-identical.

## 1. Setup and shared pieces

In [22]:
import os
GPU_ID = None   # set to 0 or 1 on a multi-GPU machine; must run before torch is imported
if GPU_ID is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_ID)

In [23]:
import math, time, glob
import torch
import torch.nn.functional as F
import pandas as pd

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '')
torch.backends.cuda.matmul.allow_tf32 = False

C = 256
V = 262_144 if DEV == 'cuda' else 8192
BUDGET, E_STOP = 0.1, 0.005
if DEV == 'cuda':
    LOGIT_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    LOGIT_DTYPE = torch.float32


def chunked_logsumexp(x, tau, chunk=32_768):
    out = None
    for s in range(0, x.shape[-1], chunk):
        part = torch.logsumexp(x[..., s:s + chunk].float() / tau, dim=-1)
        out = part if out is None else torch.logaddexp(out, part)
    return out


def exact_entropy(rows, tau):
    '''Exact entropy of each row (any leading shape), full vocabulary, fp32.'''
    lp = torch.log_softmax(rows.float() / tau, dim=-1)
    return -(lp.exp() * lp).sum(-1)


def n_commit(e, b=BUDGET):
    '''Number of slots the commit rule accepts for entropy vector e of shape [C].'''
    s, _ = torch.sort(e)
    excl = s.cumsum(0) - s
    return int((excl <= b).sum())


def commit_exact(e, b=BUDGET):
    '''Reference commit set: the n lowest-entropy slots (stable order), shape [C] bool.'''
    order = torch.argsort(e, stable=True)
    keep = torch.zeros_like(e, dtype=torch.bool)
    keep[order[:n_commit(e, b)]] = True
    return keep

device: cuda Tesla T4


## 2. Entropy intervals from the top-k

The test checks that the true entropy always lies inside [lo, hi], for both bounds.

In [24]:
NUM_EPS = 2e-4   # safety margin for float32 rounding, so [lo, hi] always contains the true entropy


def chunked_logsumexp_shifted(x, tau, shift, chunk=32_768):
    '''log-sum-exp of (x - shift) / tau, one vocabulary chunk at a time. shift is the row max, so the
    result is near 0, where float32 is precise (an unshifted value near 45 at tau=0.4 loses ~4e-6).'''
    out = None
    for s in range(0, x.shape[-1], chunk):
        part = torch.logsumexp((x[..., s:s + chunk].float() - shift) / tau, dim=-1)
        out = part if out is None else torch.logaddexp(out, part)
    return out


@torch.no_grad()
def topk_stats(logits, tau, k, tight=True, eps=NUM_EPS):
    '''Exact top-k probabilities, tail mass m and a guaranteed entropy interval [lo, hi] per slot.'''
    Vv = logits.shape[-1]
    vals, idx = torch.topk(logits, k, dim=-1)                     # sorted, largest first
    top1 = vals[..., :1].float()                                  # row max
    lse_s = chunked_logsumexp_shifted(logits, tau, top1)          # normalizer relative to the max (near 0)
    lp = (vals.float() - top1) / tau - lse_s.unsqueeze(-1)
    p = lp.exp()
    h = -(p * lp).sum(-1)
    m = (-torch.expm1(torch.logsumexp(lp, dim=-1))).clamp(0.0, 1.0)   # 1 - sum(p), computed stably
    neg_m_log_m = -m * torch.log(m.clamp_min(1e-30))
    hi = h + neg_m_log_m + (m * math.log(Vv - k) if Vv > k else 0.0)
    if tight:
        cap = torch.minimum(m, p[..., -1]).clamp_min(1e-30)       # each tail p <= smallest kept p
        lo = h - m * torch.log(cap)
    else:
        lo = h + neg_m_log_m
    lo, hi = (lo - eps).clamp_min(0.0), hi + eps                  # widen for float32 rounding error
    lse = top1.squeeze(-1) / tau + lse_s                          # unshifted normalizer, same meaning as before
    return dict(idx=idx, p=p, m=m, lo=lo, hi=hi, lse=lse)


def synth_logits(B, kind, seed=0, conf_lo=4.0, conf_span=14.0):
    g = torch.Generator(device=DEV).manual_seed(seed)
    conf = torch.rand(B, C, 1, device=DEV, generator=g) * conf_span + conf_lo
    if kind == 'uniform':
        L = torch.randn(B, C, V, device=DEV, generator=g)
    else:
        ranks = torch.arange(1, V + 1, device=DEV, dtype=torch.float32)
        perm = torch.randperm(V, device=DEV, generator=g)
        L = (-1.1 * torch.log(ranks))[perm].expand(B, C, V).clone()
        L += 0.3 * torch.randn(B, C, V, device=DEV, generator=g)
    tgt = torch.randint(V, (B, C, 1), device=DEV, generator=g)
    L.scatter_add_(-1, tgt, conf)
    return L.to(LOGIT_DTYPE)

violations = 0
for kind in ['uniform', 'zipf']:
    for tau in [0.8, 0.4]:
        for k in [16, 64, 256]:
            L = synth_logits(2, kind, seed=k)
            e = exact_entropy(L, tau)
            for tight in [False, True]:
                st = topk_stats(L, tau, k, tight)
                violations += int(((st['lo'] > e + 1e-4) | (e > st['hi'] + 1e-4)).sum())
print('bound violations:', violations)
assert violations == 0

bound violations: 0


## 3. Certified commit and certified stop

`certified_commit_row` returns exactly the reference commit set. It refines (computes exactly) only slots whose intervals make the decision ambiguous, at most `chunk` at a time. `certified_stop_row` does the same for the mean-entropy stopping test. The argmax-unchanged half of the stopping rule is already exact, since the argmax is always in the top-k.

In [25]:
@torch.no_grad()
def certified_commit_row(lo, hi, refine, b=BUDGET, chunk=8):
    '''lo, hi: [C] entropy bounds; refine(pos) -> exact entropies at positions pos.
    Returns (keep mask identical to commit_exact(true entropies), number of refined slots).'''
    lo, hi = lo.clone(), hi.clone()
    exact = torch.zeros_like(lo, dtype=torch.bool)
    refined = 0
    while True:
        if bool(exact.all()):
            return commit_exact(hi, b), refined
        n_lo, n_hi = n_commit(lo, b), n_commit(hi, b)
        if n_lo == n_hi:
            order = torch.argsort(hi, stable=True)
            S, R = order[:n_lo], order[n_lo:]
            if R.numel() == 0 or hi[S].max() < lo[R].min():
                keep = torch.zeros_like(exact)
                keep[S] = True
                return keep, refined
            amb = torch.cat([S[hi[S] >= lo[R].min()], R[lo[R] <= hi[S].max()]])
        else:
            amb = torch.argsort(lo, stable=True)[:n_lo + 1]
        amb = amb[~exact[amb]]
        if amb.numel() == 0:
            amb = (~exact).nonzero().squeeze(-1)
        if amb.numel() > chunk:
            amb = amb[torch.argsort(hi[amb] - lo[amb], descending=True)[:chunk]]
        e = refine(amb)
        lo[amb], hi[amb] = e, e
        exact[amb] = True
        refined += amb.numel()


@torch.no_grad()
def certified_stop_row(lo, hi, refine, e_stop=E_STOP, chunk=32):
    '''Decides mean(entropy) <= e_stop exactly; returns (decision, number of refined slots).'''
    lo, hi = lo.clone(), hi.clone()
    exact = torch.zeros_like(lo, dtype=torch.bool)
    refined = 0
    while True:
        if hi.mean() <= e_stop:
            return True, refined
        if lo.mean() > e_stop:
            return False, refined
        cand = (~exact).nonzero().squeeze(-1)
        if cand.numel() == 0:
            return bool(hi.mean() <= e_stop), refined
        pick = cand[torch.argsort(hi[cand] - lo[cand], descending=True)[:chunk]]
        e = refine(pick)
        lo[pick], hi[pick] = e, e
        exact[pick] = True
        refined += pick.numel()


def check_certified(L, tau, k, tight=True, e_stop=E_STOP):
    '''Run certified decisions on every sequence of L and compare with the full reference.'''
    st = topk_stats(L, tau, k, tight)
    e_true = exact_entropy(L, tau)
    out = dict(commit_mismatch=0, stop_mismatch=0, refined_commit=0, refined_stop=0)
    for bi in range(L.shape[0]):
        refine = lambda pos, bi=bi: exact_entropy(L[bi, pos], tau)
        keep, r1 = certified_commit_row(st['lo'][bi], st['hi'][bi], refine)
        stop, r2 = certified_stop_row(st['lo'][bi], st['hi'][bi], refine, e_stop)
        out['commit_mismatch'] += int((keep != commit_exact(e_true[bi])).any())
        out['stop_mismatch'] += int(stop != bool(e_true[bi].mean() <= e_stop))
        out['refined_commit'] += r1
        out['refined_stop'] += r2
    n = L.shape[0] * C
    out['refined_commit_frac'] = out.pop('refined_commit') / n
    out['refined_stop_frac'] = out.pop('refined_stop') / n
    return out

Exhaustive check on synthetic logits. Every mismatch count must be zero; the refined fractions show the cost.

In [26]:
rows = []
for kind in ['uniform', 'zipf']:
    for conf_lo in [2.0, 8.0]:
        L = synth_logits(2, kind, seed=int(conf_lo), conf_lo=conf_lo)
        for tau in [0.8, 0.6, 0.4]:
            for k in [16, 64, 256]:
                for tight in [False, True]:
                    for e_stop in [E_STOP, 0.5]:
                        r = check_certified(L, tau, k, tight, e_stop)
                        rows.append(dict(tail=kind, conf_lo=conf_lo, tau=tau, k=k, tight_lo=tight, e_stop=e_stop, **r))
        del L
cert_df = pd.DataFrame(rows)
print('total commit mismatches:', cert_df.commit_mismatch.sum(), '| total stop mismatches:', cert_df.stop_mismatch.sum())
assert cert_df.commit_mismatch.sum() == 0 and cert_df.stop_mismatch.sum() == 0
display(cert_df.groupby(['tail', 'k', 'tight_lo'])[['refined_commit_frac', 'refined_stop_frac']].mean().round(4))

total commit mismatches: 0 | total stop mismatches: 0


refined_commit_frac  refined_stop_frac
tail    k   tight_lo                                        
uniform 16  False                  0.4196             0.0521
            True                   0.3854             0.0052
        64  False                  0.4173             0.0521
            True                   0.3838             0.0052
        256 False                  0.4144             0.0417
            True                   0.3828             0.0052
zipf    16  False                  0.1387             0.0000
            True                   0.1348             0.0000
        64  False                  0.1383             0.0000
            True                   0.1341             0.0000
        256 False                  0.1341             0.0000
            True                   0.1341             0.0000

Expect the flat-tail synthetic case to need more refinement than the Zipf case: a flat tail spreads its mass over many tokens and makes the intervals wide. Real language-model tails are closer to Zipf, but only section 7 can say.

## 4. Optional: exact sampling with a tail branch

Checks on a small vocabulary that the empirical token frequencies match the true distribution (total-variation distance should be at sampling-noise level, about 0.005 here).

In [27]:
@torch.no_grad()
def sample_exact_topk(logits, tau, st, gen=None):
    '''Samples each slot from its full distribution using only the top-k, except with probability m
    (the tail mass), when that one slot's full row is used. Returns (tokens [B, C], rows that hit the tail).'''
    idx, p, m = st['idx'], st['p'], st['m']
    B, Cc, k = p.shape
    u = torch.rand(B, Cc, device=logits.device, generator=gen)
    pick = torch.multinomial(p.reshape(-1, k), 1, generator=gen).view(B, Cc)     # proportional to p within top-k
    tok = idx.gather(-1, pick.unsqueeze(-1)).squeeze(-1)
    rows = (u < m).nonzero()
    if rows.numel():
        r0, r1 = rows[:, 0], rows[:, 1]
        Lr = logits[r0, r1].float() / tau
        Lr.scatter_(-1, idx[r0, r1], float('-inf'))                              # tail tokens only
        tok[r0, r1] = torch.multinomial(torch.softmax(Lr, -1), 1, generator=gen).squeeze(-1)
    return tok, int(rows.shape[0])

Vs, tau_s, k_s, n_draws = 512, 0.8, 8, 200_000
g = torch.Generator(device=DEV).manual_seed(0)
row = torch.randn(1, 1, Vs, device=DEV, generator=g) * 2
true_p = torch.softmax(row.float() / tau_s, -1).squeeze()
L_rep = row.expand(1, n_draws, Vs).contiguous()
st = topk_stats(L_rep, tau_s, k_s)
tok, n_tail = sample_exact_topk(L_rep, tau_s, st, gen=g)
emp = torch.bincount(tok.flatten(), minlength=Vs).float() / n_draws
tv = 0.5 * (emp - true_p).abs().sum().item()
naive = torch.zeros(Vs, device=DEV)
naive[st['idx'][0, 0]] = st['p'][0, 0] / st['p'][0, 0].sum()
print(f'tail mass m = {st["m"][0, 0]:.3f} | tail branch used in {n_tail / n_draws:.3f} of draws')
print(f'TV(exact sampler, truth) = {tv:.4f}   vs   TV(plain top-k, truth) = {0.5 * (naive - true_p).abs().sum():.4f}')

tail mass m = 0.361 | tail branch used in 0.362 of draws
TV(exact sampler, truth) = 0.0090   vs   TV(plain top-k, truth) = 0.3614


## 5. Self-conditioning mix: measured error vs. the guaranteed bound

In [28]:
D_test = 512
E = torch.randn(V, D_test, device=DEV) * 0.02
E_norm_max = E.norm(dim=-1).max()
rows = []
for kind in ['uniform', 'zipf']:
    L = synth_logits(1, kind, seed=5)
    p_full = torch.softmax(L.float() / 0.6, -1)
    mix_full = p_full @ E
    for k in [16, 64, 256]:
        st = topk_stats(L, 0.6, k)
        mix_k = F.embedding_bag(st['idx'].reshape(-1, k), E, per_sample_weights=st['p'].reshape(-1, k).contiguous(),
                                mode='sum').view(1, C, -1)
        err = (mix_k - mix_full).norm(dim=-1)
        bound = st['m'] * E_norm_max
        rows.append(dict(tail=kind, k=k, mean_err=err.mean().item(), mean_bound=bound.mean().item(),
                         bound_holds=bool((err <= bound + 1e-6).all())))
    del L, p_full
display(pd.DataFrame(rows).round(6))

,tail,k,mean_err,mean_bound,bound_holds
0,uniform,16,0.000972,0.166547,True
1,uniform,64,0.000875,0.163229,True
2,uniform,256,0.000743,0.156106,True
3,zipf,16,0.002614,0.022588,True
4,zipf,64,0.000392,0.007046,True
5,zipf,256,0.000061,0.002204,True


## 6. One certified sampling step, end to end

Combines everything into the step a scheduler would run: intervals, certified commits, exact sampling, top-k mix. Returns the counters that matter for cost.

In [29]:
@torch.no_grad()
def certified_step(logits, tau, E, k=64, tight=True, gen=None, b=BUDGET):
    st = topk_stats(logits, tau, k, tight)
    B = logits.shape[0]
    keep = torch.zeros(B, logits.shape[1], dtype=torch.bool, device=logits.device)
    refined = 0
    for bi in range(B):
        refine = lambda pos, bi=bi: exact_entropy(logits[bi, pos], tau)
        keep[bi], r = certified_commit_row(st['lo'][bi], st['hi'][bi], refine, b)
        refined += r
    tok, n_tail = sample_exact_topk(logits, tau, st, gen)
    mix = F.embedding_bag(st['idx'].reshape(-1, k), E, per_sample_weights=st['p'].reshape(-1, k).contiguous(),
                          mode='sum').view(B, logits.shape[1], -1)
    return dict(keep=keep, tok=tok, mix=mix, lo=st['lo'], hi=st['hi'], m=st['m'],
                refined=refined, tail_rows=n_tail)

L = synth_logits(2, 'zipf', seed=11)
for k in [64, 256]:
    out = certified_step(L, 0.6, E, k=k)
    print(f'k={k:4d}: commits per sequence {out["keep"].sum(-1).tolist()} | refined rows {out["refined"]} '
          f'| tail-sampled rows {out["tail_rows"]} (of {2 * C})')
del L

k=  64: commits per sequence [16, 20] | refined rows 24 | tail-sampled rows 8 (of 512)
k= 256: commits per sequence [16, 20] | refined rows 24 | tail-sampled rows 0 (of 512)


## 7. The key measurement: refinement rate on real DiffusionGemma logits

Point `LOGIT_DIR` at the per-step logits saved by Notebook 2. For each step and k this reports decision mismatches (must be zero) and the fraction of slots that needed a full row. This is the number that decides whether the certified sampler is cheap in practice.

In [30]:
LOGIT_DIR = '/content/drive/MyDrive/dgemma_logits'
TAU_TABLE = None    # paste the per-step temperatures printed by Notebook 2 (list), or leave None for the linear schedule
N_STEPS, T_MAX, T_MIN = 48, 0.8, 0.4

def tau_for(step):
    if TAU_TABLE is not None:
        return TAU_TABLE[min(step, len(TAU_TABLE) - 1)]
    return T_MAX - (T_MAX - T_MIN) * step / (N_STEPS - 1)

files = sorted(glob.glob(f'{LOGIT_DIR}/step_*.pt'))
print(len(files), 'files')
rows = []
for f in files:
    L = torch.load(f).to(DEV)
    if L.dim() != 3 or L.shape[1] != C:
        print('skip', f, tuple(L.shape)); continue
    step = int(f.split('_')[-1].split('.')[0])
    tau = tau_for(step)
    for k in [16, 64, 256, 1024]:
        r = check_certified(L, tau, k, tight=True)
        rows.append(dict(step=step, tau=round(tau, 3), k=k, **r))
    del L
if rows:
    real_df = pd.DataFrame(rows)
    print('mismatches:', real_df.commit_mismatch.sum(), real_df.stop_mismatch.sum())
    display(real_df.groupby('k')[['refined_commit_frac', 'refined_stop_frac']].mean().round(4))
    display(real_df.pivot_table(index='step', columns='k', values='refined_commit_frac').round(4))

0 files


**Reading the result**
- Refinement under about 1 to 2% of slots at k = 64 or 256: the certified sampler is both exact and cheap. That is the headline claim.
- Much higher: try larger k, or a per-step k (early, high-entropy steps may need more), and report the cost curve honestly.
- The per-row Python loop here is for correctness, not speed. The batched or Triton version is the systems half of the project.